# Parameter Sensitivity Analysis - Kronos

This notebook performs a parameter sensitivity analysis on the Kronos model to evaluate the impact of different data parameters on model performance.

## Setup

In [ ]:
# Clean up any existing nested structure
!rm -rf /content/BA
print("Cleaned up any existing repositories")

In [ ]:
import os
from pathlib import Path
import os

repo_url = "https://github.com/bp571/BA"

%cd /content
!git clone {repo_url}
print("\nRepository cloned successfully")

%cd /content/BA
!pwd
print("\nWorking directory set to:", os.getcwd())

TIINGO_API_KEY = "312c6dab6a1fe6258bbc6652bcdec49a14ee08ad"

# Set environment variable for the session
os.environ["TIINGO_API_KEY"] = TIINGO_API_KEY

# Create .env file for compatibility
env_content = f"TIINGO_API_KEY={TIINGO_API_KEY}\n"
Path(".env").write_text(env_content)

print("âœ… API key configured")

# Clone Kronos model into the expected location
!git clone https://github.com/shiyu-coder/Kronos.git 02_finetuning/models/Kronos
print("\nKronos model cloned successfully")
!ls -la 02_finetuning/models/

In [ ]:
!pip install -q torch transformers peft gluonts SALib PyYAML tqdm pandas numpy matplotlib seaborn

## Configuration

In [ ]:
# Analysis Configuration
METHOD = 'grid'  # Options: 'sobol' or 'grid'
SEED = 42
BATCH_SIZE = 128
QUICK_MODE = False  # Set to True for faster testing (5 windows, 8 samples)

# Optional overrides
MAX_WINDOWS = 30  # Rolling windows pro Asset (mehr = bessere IC-Schätzung)
N_SAMPLES = 8 if QUICK_MODE else None    # Override Sobol samples

# Festes Eval-Startdatum: erste Vorhersage aller Experimente beginnt hier.
# context_steps werden aus der Pre-2021-Historie gedeckt (Daten ab 2010 verfügbar).
EVAL_START = '2021-01-01'

print(f"Method: {METHOD}")
print(f"Seed: {SEED}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Eval start date: {EVAL_START}")
print(f"Quick mode: {QUICK_MODE}")
if MAX_WINDOWS:
    print(f"Max windows: {MAX_WINDOWS}")
if N_SAMPLES:
    print(f"N samples: {N_SAMPLES}")

## Run Sensitivity Analysis

This calls the [`run_sensitivity.py`](03_sensitivity_analysis/data_parameters/run_sensitivity.py) script to execute the experiments.

In [ ]:
# Build command arguments
cmd_args = [
    f"--method {METHOD}",
    f"--seed {SEED}",
    f"--batch_size {BATCH_SIZE}",
    f"--eval-start {EVAL_START}"
]

if QUICK_MODE:
    cmd_args.append("--quick")
elif MAX_WINDOWS:
    cmd_args.append(f"--max-windows {MAX_WINDOWS}")

if N_SAMPLES and not QUICK_MODE:
    cmd_args.append(f"--n-samples {N_SAMPLES}")

cmd = f"python 03_sensitivity_analysis/data_parameters/run_sensitivity.py {' '.join(cmd_args)}"

print(f"Executing: {cmd}
")
!{cmd}

## Analyze Results

This calls the appropriate analysis script based on the method used.

In [ ]:
RESULTS_DIR = f"03_sensitivity_analysis/data_parameters/results/raw_{METHOD}"

if METHOD == 'sobol':
    # Use Sobol-specific analysis
    print("Running Sobol sensitivity analysis...\n")
    !python 03_sensitivity_analysis/data_parameters/analyze_sensitivity.py \
        --results-dir {RESULTS_DIR} \
        --visualize
elif METHOD == 'grid':
    # Use grid-specific analysis
    print("Running grid search analysis...\n")
    !python 03_sensitivity_analysis/data_parameters/analyze_grid.py \
        --results-dir {RESULTS_DIR} \
        --output-dir 03_sensitivity_analysis/data_parameters/results/
else:
    print(f"Unknown method: {METHOD}")

## Display Results

View generated plots and summaries.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pathlib import Path

if METHOD == 'sobol':
    fig_dir = Path("03_sensitivity_analysis/data_parameters/results/figures")
    
    print("Sobol Sensitivity Indices:")
    for metric in ['IC_Mean', 'RankIC_Mean', 'MAE_Mean']:
        img_path = fig_dir / f"sobol_{metric}.png"
        if img_path.exists():
            print(f"\n{metric}:")
            display(Image(filename=str(img_path)))
    
    print("\nParameter Response Curves:")
    for metric in ['IC_Mean', 'RankIC_Mean', 'MAE_Mean']:
        img_path = fig_dir / f"response_{metric}.png"
        if img_path.exists():
            print(f"\n{metric}:")
            display(Image(filename=str(img_path)))

elif METHOD == 'grid':
    fig_dir = Path("03_sensitivity_analysis/data_parameters/results/")
    
    print("Grid Search Heatmaps:")
    for metric in ['mae_mean', 'ic_mean', 'rankic_mean']:
        img_path = fig_dir / f"heatmap_{metric}.png"
        if img_path.exists():
            print(f"\n{metric.upper()}:")
            display(Image(filename=str(img_path)))

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"Results directory: {RESULTS_DIR}")
if METHOD == 'sobol':
    print("Figures: 03_sensitivity_analysis/data_parameters/results/figures/")
    print("Report: 03_sensitivity_analysis/data_parameters/results/sensitivity_report.txt")
elif METHOD == 'grid':
    print("Figures: 03_sensitivity_analysis/data_parameters/results/grid_results/")
    print("Summary: 03_sensitivity_analysis/data_parameters/results/grid_results/grid_search_summary.csv")